In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import load_npz

train = pd.read_parquet(
    "../data/processed/train.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

test = pd.read_parquet(
    "../data/processed/test.parquet"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (1928949, 5)
Validation: (413345, 5)
Test: (413347, 5)


# Identify cold users and items

In [2]:
train_users = set(
    train["user_id"].unique()
)

train_items = set(
    train["item_id"].unique()
)

test_users = set(
    test["user_id"].unique()
)

test_items = set(
    test["item_id"].unique()
)

cold_test_users = (
    test_users - train_users
)

cold_test_items = (
    test_items - train_items
)

print(
    "Training users:",
    len(train_users)
)

print(
    "Training items:",
    len(train_items)
)

print(
    "Cold test users:",
    len(cold_test_users)
)

print(
    "Cold test items:",
    len(cold_test_items)
)

Training users: 978906
Training items: 200974
Cold test users: 219888
Cold test items: 20399


# Verify the earlier cold-start numbers

In [3]:
cold_user_percentage = (
    len(cold_test_users)
    / len(test_users)
    * 100
)

cold_item_percentage = (
    len(cold_test_items)
    / len(test_items)
    * 100
)

print(
    f"Cold-user rate: "
    f"{cold_user_percentage:.2f}%"
)

print(
    f"Cold-item rate: "
    f"{cold_item_percentage:.2f}%"
)

Cold-user rate: 94.08%
Cold-item rate: 20.90%


# Build the cold-user fallback

In [4]:
popularity_ranking = (
    train
    .groupby("item_id")
    ["interaction_strength"]
    .sum()
    .sort_values(
        ascending=False
    )
    .index
    .tolist()
)

In [5]:
def cold_user_recommendations(
    k=20
):

    return popularity_ranking[:k]

In [6]:
cold_user_recs = (
    cold_user_recommendations(20)
)

print(
    "Cold-user recommendations:"
)

print(
    cold_user_recs
)

Cold-user recommendations:
[461686, 5411, 309778, 370653, 257040, 369447, 7943, 298009, 48030, 335975, 445351, 312728, 111530, 37029, 29196, 96924, 234255, 354233, 441668, 409804]


# Add diversity

In [7]:
import os

for path in [
    "../data/processed/item_categories.parquet",
    "../data/processed/item_metadata.parquet",
    "../data/processed/item_features.parquet"
]:
    print(
        path,
        "→",
        os.path.exists(path)
    )

../data/processed/item_categories.parquet → True
../data/processed/item_metadata.parquet → False
../data/processed/item_features.parquet → False


In [8]:
[x for x in globals().keys()
 if "filtered" in x.lower()
 or "metadata" in x.lower()
 or "category" in x.lower()]

[]

In [9]:
[x for x in globals().keys()
 if isinstance(
     globals()[x],
     pd.DataFrame
 )]

['train', 'validation', 'test']

In [10]:
import os

for root, dirs, files in os.walk("../data"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

../data\raw\category_tree.csv
../data\raw\events.csv
../data\raw\item_properties_part1.csv
../data\raw\item_properties_part2.csv


# Extract only categoryid

In [11]:
import pandas as pd
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

property_files = [
    "../data/raw/item_properties_part1.csv",
    "../data/raw/item_properties_part2.csv"
]

category_frames = []

for file in property_files:

    print(f"Reading: {file}")

    chunks = []

    for chunk in pd.read_csv(
        file,
        usecols=["itemid", "property", "value"],
        chunksize=500_000
    ):

        category_chunk = chunk[
            chunk["property"].astype(str)
            == "categoryid"
        ][
            ["itemid", "value"]
        ].copy()

        category_chunk.rename(
            columns={
                "value": "categoryid"
            },
            inplace=True
        )

        chunks.append(category_chunk)

    part_categories = pd.concat(
        chunks,
        ignore_index=True
    )

    print(
        "Category rows:",
        len(part_categories)
    )

    category_frames.append(
        part_categories
    )

Reading: ../data/raw/item_properties_part1.csv
Category rows: 426305
Reading: ../data/raw/item_properties_part2.csv
Category rows: 361909


# Combine and clean

In [12]:
item_categories = pd.concat(
    category_frames,
    ignore_index=True
)

item_categories["categoryid"] = pd.to_numeric(
    item_categories["categoryid"],
    errors="coerce"
)

item_categories = (
    item_categories
    .dropna(subset=["categoryid"])
)

item_categories["categoryid"] = (
    item_categories["categoryid"]
    .astype(int)
)

item_categories = (
    item_categories
    .drop_duplicates(
        subset=["itemid", "categoryid"]
    )
)

print(
    "Unique item-category pairs:",
    len(item_categories)
)

print(
    "Unique items:",
    item_categories["itemid"].nunique()
)

print(
    "Unique categories:",
    item_categories["categoryid"].nunique()
)

print(
    item_categories.head()
)

Unique item-category pairs: 442432
Unique items: 417053
Unique categories: 1242
   itemid  categoryid
0  460429        1338
1  281245        1277
2   35575        1059
3    8313        1147
4   55102          47


In [13]:
item_categories.to_parquet(
    "../data/processed/item_categories.parquet",
    index=False
)

print(
    "Saved:",
    "../data/processed/item_categories.parquet"
)

Saved: ../data/processed/item_categories.parquet


In [14]:
category_file = pd.read_parquet(
    "../data/processed/item_categories.parquet"
)

print(
    category_file.shape
)

print(
    category_file.head(10)
)

(442432, 2)
   itemid  categoryid
0  460429        1338
1  281245        1277
2   35575        1059
3    8313        1147
4   55102          47
5  397079         619
6  265036        1228
7  124459        1277
8  350508         546
9  221365        1226


## category-diversified cold-user recommender.

# Load category data

In [15]:
item_categories = pd.read_parquet(
    "../data/processed/item_categories.parquet"
)

print(item_categories.shape)

(442432, 2)


# Connect categories with popularity

In [16]:
# Popularity calculated strictly from TRAINING data

popularity_scores = (
    train
    .groupby("item_id")["interaction_strength"]
    .sum()
    .sort_values(ascending=False)
)

print("Popular items:", len(popularity_scores))
print(popularity_scores.head(20))

Popular items: 200974
item_id
461686    2142
5411      1903
309778    1754
370653    1483
257040    1477
369447    1475
7943      1443
298009    1347
48030     1268
335975    1233
445351    1216
312728    1203
111530    1186
37029     1175
29196     1130
96924     1097
234255    1048
354233    1020
441668    1004
409804     956
Name: interaction_strength, dtype: int64


In [17]:
popular_items = (
    popularity_scores
    .rename("popularity")
    .reset_index()
)

popular_items.rename(
    columns={
        "item_id": "itemid"
    },
    inplace=True
)

print(popular_items.head())

   itemid  popularity
0  461686        2142
1    5411        1903
2  309778        1754
3  370653        1483
4  257040        1477


# Add category information

In [18]:
popular_with_category = popular_items.merge(
    item_categories,
    on="itemid",
    how="left"
)

print(
    popular_with_category.shape
)

print(
    popular_with_category.head()
)

(213614, 3)
   itemid  popularity  categoryid
0  461686        2142      1037.0
1    5411        1903       789.0
2  309778        1754       683.0
3  370653        1483        82.0
4  257040        1477       683.0


In [19]:
print(
    "Popular items with category:",
    popular_with_category["categoryid"]
    .notna()
    .mean()
)

Popular items with category: 0.8113232278783226


# Build the diversified recommender

In [20]:
def diversified_cold_user_recommendations(
    k=20,
    max_per_category=2
):

    recommendations = []
    category_counts = {}

    for _, row in popular_with_category.iterrows():

        item_id = int(
            row["itemid"]
        )

        category = row["categoryid"]

        # If category unavailable
        if pd.isna(category):

            recommendations.append(
                item_id
            )

        else:

            category = int(category)

            current_count = (
                category_counts
                .get(category, 0)
            )

            if (
                current_count
                >= max_per_category
            ):
                continue

            recommendations.append(
                item_id
            )

            category_counts[category] = (
                current_count + 1
            )

        if len(recommendations) >= k:
            break

    return recommendations[:k]

In [21]:
cold_recs = (
    diversified_cold_user_recommendations(
        k=20,
        max_per_category=2
    )
)

print(
    "Cold-user recommendations:"
)

print(cold_recs)

Cold-user recommendations:
[461686, 5411, 309778, 370653, 257040, 369447, 7943, 298009, 48030, 335975, 445351, 312728, 111530, 37029, 29196, 96924, 234255, 354233, 354233, 441668]


# Inspect category diversity

In [22]:
cold_rec_categories = (
    item_categories[
        item_categories["itemid"]
        .isin(cold_recs)
    ]
)

print(
    cold_rec_categories
)

        itemid  categoryid
11194   111530        1625
33489    48030        1219
102165  441668        1263
106666  369447          48
139066  370653          82
161262  312728        1098
187942  298009         529
197315   29196        1265
204511  354233        1613
214867   37029        1483
218602    5411         789
243772  354233         491
245695   96924          56
254178  234255        1051
289478  461686        1037
319853  309778         683
386243    7943         398
389679  335975          48
411956  257040         683
414734  445351        1483


In [23]:
print(
    "Unique categories:",
    cold_rec_categories[
        "categoryid"
    ].nunique()
)

Unique categories: 17


# Compare the two strategies

In [24]:
baseline_cold_recs = (
    popularity_ranking[:20]
)

In [25]:
diverse_cold_recs = (
    diversified_cold_user_recommendations(
        k=20,
        max_per_category=2
    )
)

In [26]:
print(
    "Popularity baseline:"
)

print(
    baseline_cold_recs
)

print(
    "\nDiversified:"
)

print(
    diverse_cold_recs
)

Popularity baseline:
[461686, 5411, 309778, 370653, 257040, 369447, 7943, 298009, 48030, 335975, 445351, 312728, 111530, 37029, 29196, 96924, 234255, 354233, 441668, 409804]

Diversified:
[461686, 5411, 309778, 370653, 257040, 369447, 7943, 298009, 48030, 335975, 445351, 312728, 111530, 37029, 29196, 96924, 234255, 354233, 354233, 441668]


In [27]:
print(
    "Baseline categories:",
    item_categories[
        item_categories["itemid"]
        .isin(baseline_cold_recs)
    ]["categoryid"].nunique()
)

print(
    "Diversified categories:",
    item_categories[
        item_categories["itemid"]
        .isin(diverse_cold_recs)
    ]["categoryid"].nunique()
)

print(
    "Baseline length:",
    len(baseline_cold_recs)
)

print(
    "Diversified length:",
    len(diverse_cold_recs)
)

Baseline categories: 18
Diversified categories: 17
Baseline length: 20
Diversified length: 20


In [28]:
cold_test_ground_truth = (
    test[
        test["user_id"].isin(
            cold_test_users
        )
    ]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

cold_eval_users = np.array(
    sorted(
        cold_test_ground_truth.keys()
    )
)

print(
    "Cold users with test interactions:",
    len(cold_eval_users)
)

print(
    "Cold-user test interactions:",
    sum(
        len(items)
        for items in
        cold_test_ground_truth.values()
    )
)

Cold users with test interactions: 219888
Cold-user test interactions: 300162


In [29]:
def evaluate_cold_recommendations(
    recommendations,
    ground_truth,
    users,
    ks=(5, 10, 20)
):

    results = {
        k: {
            "precision": [],
            "recall": [],
            "ndcg": [],
            "hit": []
        }
        for k in ks
    }

    for user_id in users:

        relevant = ground_truth[
            user_id
        ]

        for k in ks:

            recs = recommendations[:k]

            results[k]["precision"].append(
                precision_at_k(
                    recs,
                    relevant,
                    k
                )
            )

            results[k]["recall"].append(
                recall_at_k(
                    recs,
                    relevant,
                    k
                )
            )

            results[k]["ndcg"].append(
                ndcg_at_k(
                    recs,
                    relevant,
                    k
                )
            )

            results[k]["hit"].append(
                hit_rate_at_k(
                    recs,
                    relevant,
                    k
                )
            )

    return pd.DataFrame([
        {
            "K": k,
            "Precision@K":
                np.mean(
                    results[k]["precision"]
                ),
            "Recall@K":
                np.mean(
                    results[k]["recall"]
                ),
            "NDCG@K":
                np.mean(
                    results[k]["ndcg"]
                ),
            "HitRate@K":
                np.mean(
                    results[k]["hit"]
                )
        }
        for k in ks
    ])

In [30]:
def precision_at_k(recommended, relevant, k):
    hits = sum(
        item in relevant
        for item in recommended[:k]
    )
    return hits / k if k > 0 else 0.0


def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / len(relevant)


def hit_rate_at_k(recommended, relevant, k):
    return float(
        any(
            item in relevant
            for item in recommended[:k]
        )
    )


def ndcg_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0

    dcg = 0.0

    for rank, item in enumerate(
        recommended[:k],
        start=1
    ):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(
        len(relevant),
        k
    )

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    return dcg / idcg

In [31]:
cold_popularity_results = (
    evaluate_cold_recommendations(
        baseline_cold_recs,
        cold_test_ground_truth,
        cold_eval_users
    )
)

cold_popularity_results.insert(
    0,
    "Model",
    "Cold-Popularity"
)

print(cold_popularity_results)

             Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Cold-Popularity   5     0.000566  0.002252  0.001758   0.002829
1  Cold-Popularity  10     0.000435  0.003573  0.002177   0.004293
2  Cold-Popularity  20     0.000410  0.006374  0.002913   0.008045


In [32]:
cold_diverse_results = (
    evaluate_cold_recommendations(
        diverse_cold_recs,
        cold_test_ground_truth,
        cold_eval_users
    )
)

cold_diverse_results.insert(
    0,
    "Model",
    "Cold-Diversified"
)

print(cold_diverse_results)

              Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Cold-Diversified   5     0.000566  0.002252  0.001758   0.002829
1  Cold-Diversified  10     0.000435  0.003573  0.002177   0.004293
2  Cold-Diversified  20     0.000399  0.006263  0.002882   0.007849


In [33]:
cold_start_results = pd.concat(
    [
        cold_popularity_results,
        cold_diverse_results
    ],
    ignore_index=True
)

print(cold_start_results)

              Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0   Cold-Popularity   5     0.000566  0.002252  0.001758   0.002829
1   Cold-Popularity  10     0.000435  0.003573  0.002177   0.004293
2   Cold-Popularity  20     0.000410  0.006374  0.002913   0.008045
3  Cold-Diversified   5     0.000566  0.002252  0.001758   0.002829
4  Cold-Diversified  10     0.000435  0.003573  0.002177   0.004293
5  Cold-Diversified  20     0.000399  0.006263  0.002882   0.007849


In [34]:
cold_test_items_in_test = (
    set(test["item_id"].unique())
    - set(train["item_id"].unique())
)

print("Cold test items:", len(cold_test_items_in_test))
print(
    "Cold item rate:",
    f"{len(cold_test_items_in_test) / test['item_id'].nunique() * 100:.2f}%"
)

Cold test items: 20399
Cold item rate: 20.90%


In [35]:
cold_item_test_interactions = test[
    test["item_id"].isin(cold_test_items_in_test)
]

print(
    "Interactions with cold items:",
    len(cold_item_test_interactions)
)

print(
    "Users interacting with cold items:",
    cold_item_test_interactions["user_id"].nunique()
)

Interactions with cold items: 44758
Users interacting with cold items: 28570


In [36]:
cold_item_ground_truth = (
    cold_item_test_interactions
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

cold_item_eval_users = np.array(
    sorted(cold_item_ground_truth.keys())
)

print(
    "Cold-item evaluation users:",
    len(cold_item_eval_users)
)

print(
    "Cold-item ground-truth interactions:",
    sum(
        len(v)
        for v in cold_item_ground_truth.values()
    )
)

Cold-item evaluation users: 28570
Cold-item ground-truth interactions: 35951


In [37]:
import numpy as np
from scipy.sparse import load_npz

# Load Content-Based product matrix
product_matrix = load_npz(
    "../models/product_tfidf.npz"
)

# Load corresponding product IDs
product_ids = np.load(
    "../models/product_ids.npy"
)

print(
    "Product matrix:",
    product_matrix.shape
)

print(
    "Product IDs:",
    len(product_ids)
)

print(
    "First 10 product IDs:",
    product_ids[:10]
)

Product matrix: (160670, 86490)
Product IDs: 160670
First 10 product IDs: [ 4  6 15 16 17 19 22 24 25 26]


In [38]:
# Cold-item candidate IDs that exist in our content model
cold_items_with_content = (
    cold_test_items_in_test
    & set(product_ids)
)

print(
    "Cold items with content representation:",
    len(cold_items_with_content)
)

Cold items with content representation: 0


In [39]:
cold_item_indices = [
    product_id_to_index[item_id]
    for item_id in cold_items_with_content
]

cold_item_ids = np.array([
    product_ids[idx]
    for idx in cold_item_indices
])

print(
    "Content-available cold items:",
    len(cold_item_ids)
)

Content-available cold items: 0


In [40]:
user_profiles = load_npz(
    "../models/user_profiles.npz"
)

warm_user_ids = np.load(
    "../models/warm_user_ids.npy"
)

warm_user_to_index = {
    int(user_id): idx
    for idx, user_id in enumerate(
        warm_user_ids
    )
}

print(
    "User profiles:",
    user_profiles.shape
)

print(
    "Warm users:",
    len(warm_user_ids)
)

User profiles: (18965, 86490)
Warm users: 18965


In [41]:
print("product_matrix:", product_matrix.shape)
print("product_ids:", len(product_ids))

print(
    "item_categories:",
    item_categories.shape
)

product_matrix: (160670, 86490)
product_ids: 160670
item_categories: (442432, 2)


In [42]:
import os

paths = [
    "../models/product_tfidf.npz",
    "../models/product_ids.npy",
    "../data/processed/item_categories.parquet"
]

for path in paths:
    print(path, "→", os.path.exists(path))

../models/product_tfidf.npz → True
../models/product_ids.npy → True
../data/processed/item_categories.parquet → True


In [43]:
from scipy.sparse import csr_matrix

# Map all categories to feature indices
category_ids = np.sort(
    item_categories["categoryid"]
    .unique()
)

category_to_index = {
    int(cat): idx
    for idx, cat in enumerate(category_ids)
}

print(
    "Category feature dimensions:",
    len(category_to_index)
)

Category feature dimensions: 1242


## Build the all-item category matrix

In [44]:
from scipy.sparse import csr_matrix

all_item_ids = np.sort(
    item_categories["itemid"].unique()
)

item_to_row = {
    int(item_id): idx
    for idx, item_id in enumerate(all_item_ids)
}

rows = []
cols = []

for row in item_categories.itertuples(index=False):

    item_id = int(row.itemid)
    category_id = int(row.categoryid)

    if (
        item_id in item_to_row
        and category_id in category_to_index
    ):
        rows.append(
            item_to_row[item_id]
        )

        cols.append(
            category_to_index[category_id]
        )

data = np.ones(
    len(rows),
    dtype=np.float32
)

all_item_category_matrix = csr_matrix(
    (
        data,
        (rows, cols)
    ),
    shape=(
        len(all_item_ids),
        len(category_to_index)
    ),
    dtype=np.float32
)

print(
    "All-item matrix:",
    all_item_category_matrix.shape
)

print(
    "Non-zero values:",
    all_item_category_matrix.nnz
)

print(
    "Catalog items:",
    len(all_item_ids)
)

All-item matrix: (417053, 1242)
Non-zero values: 442432
Catalog items: 417053


# Verify cold items are actually represented

In [45]:
cold_item_indices_all = [
    item_to_row[item_id]
    for item_id in cold_test_items_in_test
    if item_id in item_to_row
]

print(
    "Cold items with category representation:",
    len(cold_item_indices_all)
)

Cold items with category representation: 14951


In [46]:
cold_item_ids_all = np.array([
    all_item_ids[idx]
    for idx in cold_item_indices_all
])

print(
    "Cold item IDs:",
    len(cold_item_ids_all)
)

print(
    cold_item_ids_all[:20]
)

Cold item IDs: 14951
[     3 262178     41 131115 262191 262193 131139 131147 393301 131158
 262232 393314 393323 131181 131182    117    125 262276 262300 393378]


# Save the cold-item artifact

In [47]:
import os

os.makedirs(
    "../models/cold_start",
    exist_ok=True
)

from scipy.sparse import save_npz

save_npz(
    "../models/cold_start/all_item_categories.npz",
    all_item_category_matrix
)

np.save(
    "../models/cold_start/all_item_ids.npy",
    all_item_ids
)

print("Cold-item content artifacts saved.")

Cold-item content artifacts saved.


## Build the cold-item category mapping

In [48]:
cold_item_to_category = (
    item_categories[
        item_categories["itemid"].isin(
            cold_item_ids_all
        )
    ]
    .groupby("itemid")["categoryid"]
    .apply(set)
    .to_dict()
)

print(
    "Cold items with category mapping:",
    len(cold_item_to_category)
)

Cold items with category mapping: 14951


# Build a user category profile

In [49]:
train_with_categories = train.merge(
    item_categories,
    left_on="item_id",
    right_on="itemid",
    how="inner"
)

user_category_counts = (
    train_with_categories
    .groupby(
        ["user_id", "categoryid"]
    )["interaction_strength"]
    .sum()
)

print(
    "User-category pairs:",
    len(user_category_counts)
)

User-category pairs: 1090074


In [50]:
user_category_profile = (
    user_category_counts
    .groupby(level=0)
    .apply(
        lambda x:
        x.sort_values(
            ascending=False
        ).index.get_level_values(
            "categoryid"
        ).tolist()
    )
    .to_dict()
)

print(
    "Users with category profiles:",
    len(user_category_profile)
)

Users with category profiles: 859881


# Cold-item recommendation function

In [51]:
def recommend_cold_items_for_user(
    user_id,
    k=10
):

    preferred_categories = (
        user_category_profile.get(
            user_id,
            []
        )
    )

    candidates = []

    for item_id in cold_item_ids_all:

        categories = (
            cold_item_to_category.get(
                int(item_id),
                set()
            )
        )

        if not categories:
            continue

        overlap = (
            len(
                set(preferred_categories)
                & categories
            )
        )

        if overlap > 0:
            candidates.append(
                (
                    item_id,
                    overlap
                )
            )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        item_id
        for item_id, score
        in candidates[:k]
    ]

# Test it on a known user

In [52]:
sample_user = next(
    iter(user_category_profile)
)

sample_recommendations = (
    recommend_cold_items_for_user(
        sample_user,
        k=10
    )
)

print(
    "Sample user:",
    sample_user
)

print(
    "Cold-item recommendations:",
    sample_recommendations
)

Sample user: 3
Cold-item recommendations: [3, 8809, 272037, 403570, 167183, 168618, 302384, 303241, 173927, 177582]


In [53]:
def recommend_cold_items_for_user(
    user_id,
    k=10
):

    preferred_categories = set(
        user_category_profile.get(
            user_id,
            []
        )
    )

    # Items already seen by user
    seen_items = set(
        train.loc[
            train["user_id"] == user_id,
            "item_id"
        ]
    )

    candidates = []

    for item_id in cold_item_ids_all:

        item_id = int(item_id)

        # Never recommend training items
        if item_id in seen_items:
            continue

        categories = cold_item_to_category.get(
            item_id,
            set()
        )

        if not categories:
            continue

        overlap = len(
            preferred_categories
            & categories
        )

        if overlap > 0:
            candidates.append(
                (item_id, overlap)
            )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        item_id
        for item_id, score
        in candidates[:k]
    ]

In [54]:
sample_recommendations = (
    recommend_cold_items_for_user(
        3,
        k=10
    )
)

print(
    sample_recommendations
)

[3, 8809, 272037, 403570, 167183, 168618, 302384, 303241, 173927, 177582]


In [55]:
seen = set(
    train.loc[
        train["user_id"] == 3,
        "item_id"
    ]
)

print(
    "Overlap:",
    set(sample_recommendations) & seen
)

Overlap: set()


In [56]:
def evaluate_cold_item_recommendations(
    users,
    ground_truth,
    k=10
):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_scores = []

    evaluated = 0

    for user_id in users:

        # Content/category profile must exist
        if user_id not in user_category_profile:
            continue

        recommended = (
            recommend_cold_items_for_user(
                user_id,
                k=k
            )
        )

        if not recommended:
            continue

        relevant = ground_truth[user_id]

        precision_scores.append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

        evaluated += 1

        if evaluated % 1000 == 0:
            print(
                f"Evaluated {evaluated} users"
            )

    return {
        "Model": "Cold-Item Category",
        "K": k,
        "Precision@K": np.mean(
            precision_scores
        ),
        "Recall@K": np.mean(
            recall_scores
        ),
        "NDCG@K": np.mean(
            ndcg_scores
        ),
        "HitRate@K": np.mean(
            hit_scores
        ),
        "Users Evaluated": evaluated
    }

In [57]:
cold_item_results = []

for k in [5, 10, 20]:

    result = (
        evaluate_cold_item_recommendations(
            cold_item_eval_users,
            cold_item_ground_truth,
            k=k
        )
    )

    cold_item_results.append(result)

cold_item_results = pd.DataFrame(
    cold_item_results
)

print(cold_item_results)

Evaluated 1000 users
Evaluated 1000 users
Evaluated 1000 users
                Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K  \
0  Cold-Item Category   5     0.003323  0.011339  0.007663   0.015106   
1  Cold-Item Category  10     0.002870  0.020534  0.010676   0.024673   
2  Cold-Item Category  20     0.002492  0.035377  0.014631   0.042296   

   Users Evaluated  
0             1986  
1             1986  
2             1986  


In [58]:
import os

os.makedirs(
    "../models/evaluation",
    exist_ok=True
)

In [59]:
import os
import pandas as pd

os.makedirs(
    "../models/evaluation",
    exist_ok=True
)

final_test_comparison = pd.DataFrame([
    {
        "Model": "Hybrid",
        "Precision@10": 0.007518,
        "Recall@10": 0.016739,
        "NDCG@10": 0.015367,
        "HitRate@10": 0.055319
    },
    {
        "Model": "Content-Based",
        "Precision@10": 0.005532,
        "Recall@10": 0.016859,
        "NDCG@10": 0.011305,
        "HitRate@10": 0.043972
    },
    {
        "Model": "BPR",
        "Precision@10": 0.002411,
        "Recall@10": 0.003171,
        "NDCG@10": 0.004455,
        "HitRate@10": 0.018440
    },
    {
        "Model": "Popularity",
        "Precision@10": 0.001560,
        "Recall@10": 0.000167,
        "NDCG@10": 0.002273,
        "HitRate@10": 0.012766
    }
])

final_test_comparison.to_csv(
    "../models/evaluation/final_test_results.csv",
    index=False
)

print(final_test_comparison)
print("\nSaved successfully.")

           Model  Precision@10  Recall@10   NDCG@10  HitRate@10
0         Hybrid      0.007518   0.016739  0.015367    0.055319
1  Content-Based      0.005532   0.016859  0.011305    0.043972
2            BPR      0.002411   0.003171  0.004455    0.018440
3     Popularity      0.001560   0.000167  0.002273    0.012766

Saved successfully.


In [60]:
cold_start_results.to_csv(
    "../models/evaluation/cold_user_results.csv",
    index=False
)

In [61]:
cold_item_results.to_csv(
    "../models/evaluation/cold_item_results.csv",
    index=False
)

In [62]:
hybrid_config = pd.DataFrame([
    {
        "content_weight": 0.4,
        "bpr_weight": 0.4,
        "popularity_weight": 0.2,
        "validation_ndcg_at_10": 0.019391,
        "test_ndcg_at_10": 0.015367
    }
])

hybrid_config.to_csv(
    "../models/evaluation/hybrid_config.csv",
    index=False
)

In [63]:
print("cold_test_items_in_test:", len(cold_test_items_in_test))
print("cold_item_ids:", len(cold_item_ids))
print("first 20:", list(cold_item_ids)[:20])

cold_test_items_in_test: 20399
cold_item_ids: 0
first 20: []


In [64]:
cold_test_items_in_test = (
    set(test["item_id"].unique())
    - set(train["item_id"].unique())
)

all_item_ids = np.sort(
    item_categories["itemid"].unique()
)

item_to_row = {
    int(item_id): idx
    for idx, item_id in enumerate(all_item_ids)
}

cold_item_indices_all = [
    item_to_row[item_id]
    for item_id in cold_test_items_in_test
    if item_id in item_to_row
]

cold_item_ids_all = np.array([
    all_item_ids[idx]
    for idx in cold_item_indices_all
])

print("Cold test items:", len(cold_test_items_in_test))
print("Cold item IDs:", len(cold_item_ids_all))
print(cold_item_ids_all[:20])

Cold test items: 20399
Cold item IDs: 14951
[     3 262178     41 131115 262191 262193 131139 131147 393301 131158
 262232 393314 393323 131181 131182    117    125 262276 262300 393378]


In [65]:
np.save(
    "../models/cold_start/cold_item_ids.npy",
    cold_item_ids_all
)

print("Saved:", len(cold_item_ids_all))

Saved: 14951
